# Subtype models: Longformer optimized for accuracy

Trains Longformer classifiers for the stigma subtypes using accuracy-based selection.

This notebook accompanies [Stigmatizing Language in Gender-Expansive Patient Records: Corpus Development, Disparity Analysis, and Natural Language Processing-Based Detection Study](https://www.jmir.org/2026/1/e91089).

## Data and execution requirements

- Clinical note text and MIMIC identifiers are not included in this repository.
- Run this notebook only in an environment authorized to access MIMIC-IV and the credentialed annotation release.
- Set `GEP_DATA_DIR`, `GEP_MODEL_DIR`, `GEP_RESULTS_DIR`, and `GEP_FIGURES_DIR` as needed. By default, repository-local directories are used.
- The notebook outputs and execution counters have been removed from the public version.

**Selection protocol.** For each subtype, the target metric is optimized on a stratified internal split of the official training set. The held-out testing set does not determine an epoch, model, hyperparameter, or decision threshold.


In [ ]:
# Repository-local path configuration
from pathlib import Path
import os

PROJECT_ROOT = Path(os.environ.get("GEP_PROJECT_ROOT", Path.cwd())).resolve()
DATA_DIR = Path(os.environ.get("GEP_DATA_DIR", PROJECT_ROOT / "data")).resolve()
MODEL_DIR = Path(os.environ.get("GEP_MODEL_DIR", PROJECT_ROOT / "models")).resolve()
RESULTS_DIR = Path(os.environ.get("GEP_RESULTS_DIR", PROJECT_ROOT / "results")).resolve()
FIGURES_DIR = Path(os.environ.get("GEP_FIGURES_DIR", PROJECT_ROOT / "figures")).resolve()

for directory in (MODEL_DIR, RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)


In [ ]:
# ==== Imports ====
import os
import numpy as np
import pandas as pd
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from threshold_protocol import make_internal_selection_split
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix
from transformers import (
    LongformerForSequenceClassification,
    LongformerTokenizer,
    get_linear_schedule_with_warmup
)

# ==== Configuration ====

# ==== Config ====
BASE_SAVE_DIR = str(MODEL_DIR / 'Longformer_GEP_SUBTYPES')
os.makedirs(BASE_SAVE_DIR, exist_ok=True)

TRAIN_PATH = str(DATA_DIR / 'GEP_train_80_20.csv')

MODEL_NAME = "allenai/longformer-base-4096"
BATCH_SIZE = 8
NUM_EPOCHS = 20
LR = 1e-5
MAX_LENGTH = 4096
USE_GLOBAL_ATTENTION = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==== Load Data ====
official_train_df = pd.read_csv(TRAIN_PATH)

subtypes = ['Credibility and Obstinacy', 'Compliance', 'Descriptors', 'Misgendering']

# ==== Dataset Class ====
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=4096, use_global_attention=False):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.use_global_attention = use_global_attention

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = float(self.labels[idx])

        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=self.max_length
        )
        input_ids = inputs["input_ids"].squeeze(0)
        attention_mask = inputs["attention_mask"].squeeze(0)

        output = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": torch.tensor(label, dtype=torch.float)
        }

        if self.use_global_attention:
            global_attention_mask = torch.zeros_like(input_ids)
            global_attention_mask[0] = 1
            output["global_attention_mask"] = global_attention_mask

        return output

def custom_collate_fn(batch):
    input_ids = torch.stack([b['input_ids'] for b in batch])
    attention_mask = torch.stack([b['attention_mask'] for b in batch])
    labels = torch.tensor([b['labels'] for b in batch], dtype=torch.float)
    global_attention_mask = None
    if 'global_attention_mask' in batch[0]:
        global_attention_mask = torch.stack([b['global_attention_mask'] for b in batch])
    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels,
        'global_attention_mask': global_attention_mask
    }

# ==== Training & Evaluation ====
def train_one_epoch(model, data_loader, criterion, optimizer, device, scheduler):
    model.train()
    total_loss, correct, total = 0, 0, 0
    pbar = tqdm(data_loader, desc="Training")
    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        global_attention_mask = batch.get("global_attention_mask")
        if global_attention_mask is not None:
            global_attention_mask = global_attention_mask.to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            global_attention_mask=global_attention_mask
        )
        loss = criterion(outputs.logits.view(-1), labels.view(-1))
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * labels.size(0)
        preds = (torch.sigmoid(outputs.logits.view(-1)) >= 0.5).float()
        correct += (preds == labels.view(-1)).sum().item()
        total += labels.size(0)

        pbar.set_postfix({'loss': total_loss / total, 'acc': 100 * correct / total})
    return total_loss / total, 100 * correct / total

def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    pbar = tqdm(data_loader, desc="Validating")
    with torch.no_grad():
        for batch in pbar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            global_attention_mask = batch.get("global_attention_mask")
            if global_attention_mask is not None:
                global_attention_mask = global_attention_mask.to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                global_attention_mask=global_attention_mask
            )
            loss = criterion(outputs.logits.view(-1), labels.view(-1))
            total_loss += loss.item() * labels.size(0)

            probs = torch.sigmoid(outputs.logits.view(-1))
            preds = (probs >= 0.5).float()

            correct += (preds == labels.view(-1)).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            pbar.set_postfix({'val_loss': total_loss / total, 'val_acc': 100 * correct / total})
    return total_loss / total, 100 * correct / total, np.array(all_preds), np.array(all_labels)

def get_predictions(model, data_loader, device):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            global_attention_mask = batch.get("global_attention_mask")
            if global_attention_mask is not None:
                global_attention_mask = global_attention_mask.to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, global_attention_mask=global_attention_mask)
            probs = torch.sigmoid(outputs.logits.view(-1))
            pred = (probs >= 0.5).float()
            preds.extend(pred.cpu().numpy())
            labels.extend(batch["labels"].cpu().numpy())
    return np.array(preds), np.array(labels)

# ==== Sequential Training ====
for subtype in subtypes:
    print("\n" + "="*90)
    print(f" Training Longformer for subtype: {subtype}")
    print("="*90)

    train_df, selection_df = make_internal_selection_split(
        official_train_df, label_column=subtype, selection_fraction=0.15, random_state=42
    )
    train_texts = train_df["text"].astype(str).tolist()
    val_texts = selection_df["text"].astype(str).tolist()
    train_labels = train_df[subtype].astype(int).tolist()
    val_labels = selection_df[subtype].astype(int).tolist()

    tokenizer = LongformerTokenizer.from_pretrained(MODEL_NAME)
    model = LongformerForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).to(DEVICE)

    train_dataset = TextDataset(train_texts, train_labels, tokenizer, max_length=MAX_LENGTH, use_global_attention=USE_GLOBAL_ATTENTION)
    val_dataset   = TextDataset(val_texts, val_labels, tokenizer, max_length=MAX_LENGTH, use_global_attention=USE_GLOBAL_ATTENTION)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=custom_collate_fn)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=custom_collate_fn)

    optimizer = optim.AdamW(model.parameters(), lr=LR)
    criterion = nn.BCEWithLogitsLoss()
    total_steps = len(train_loader) * NUM_EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    SAVE_DIR = os.path.join(BASE_SAVE_DIR, subtype.replace(" ", "_"))
    os.makedirs(SAVE_DIR, exist_ok=True)

    best_val_acc = 0.0
    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\n===== Epoch {epoch}/{NUM_EPOCHS} for {subtype} =====")
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, scheduler)
        val_loss, val_acc, val_preds, val_labels_np = evaluate(model, val_loader, criterion, DEVICE)

        print(f"[{subtype}] Epoch {epoch}: Train Acc={train_acc:.2f}% | Val Acc={val_acc:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            model.save_pretrained(SAVE_DIR)
            tokenizer.save_pretrained(SAVE_DIR)
            print(f" Saved best model for {subtype} (Val Acc={best_val_acc:.2f}%) → {SAVE_DIR}")

            cm = confusion_matrix(val_labels_np, val_preds)
            print("Confusion Matrix:")
            print(cm)
            print("Classification Report:")
            print(classification_report(val_labels_np, val_preds, digits=3))

print("\n All Longformer subtype trainings completed successfully.")
